In [3]:
%pip install -q "psycopg[binary]"


Note: you may need to restart the kernel to use updated packages.


In [4]:
from pathlib import Path

import os
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data" / "olist"
DATA_DIR.mkdir(parents=True, exist_ok=True)

pg = pgserver.get_server(str(PROJECT_DIR / "hw4_pg"), cleanup_mode="stop")

engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    version = conn.execute(text("SELECT version()")).scalar_one()
    print(version)


PostgreSQL 16.2 on aarch64-apple-darwin23.5.0, compiled by Apple clang version 15.0.0 (clang-1500.0.40.1), 64-bit


In [6]:
%load_ext sql
%config SqlMagic.autopandas = True

db_url = pg.get_uri()
%sql $db_url


The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [8]:
%%sql

SELECT
    current_database() AS database_name,
    current_timestamp AS current_time;

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-04%2Fhw4_pg
1 rows affected.


,database_name,current_time
0,postgres,2026-11-01 17:07:30.295360+01:00


Завантаження Olist із Kaggle

In [14]:
%pip install -q kagglehub


Note: you may need to restart the kernel to use updated packages.


In [21]:
import kagglehub
import os

dataset_dir = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

print("Dataset directory:", dataset_dir)
print("Files:", os.listdir(dataset_dir))


Dataset directory: /Users/macbook/.cache/kagglehub/datasets/olistbr/brazilian-ecommerce/versions/2
Files: ['olist_sellers_dataset.csv', 'product_category_name_translation.csv', 'olist_orders_dataset.csv', 'olist_order_items_dataset.csv', 'olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_products_dataset.csv']


In [22]:
files = {
    "olist_customers_raw": "olist_customers_dataset.csv",
    "olist_orders_raw": "olist_orders_dataset.csv",
    "olist_order_items_raw": "olist_order_items_dataset.csv",
    "olist_products_raw": "olist_products_dataset.csv",
    "olist_sellers_raw": "olist_sellers_dataset.csv",
    "olist_order_reviews_raw": "olist_order_reviews_dataset.csv",
}

for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)

    df = pd.read_csv(path)

    df.to_sql(table_name, engine, if_exists="replace", index=False, chunksize=10_000)

    print(table_name, df.shape)


olist_customers_raw (99441, 5)
olist_orders_raw (99441, 8)
olist_order_items_raw (112650, 7)
olist_products_raw (32951, 9)
olist_sellers_raw (3095, 4)
olist_order_reviews_raw (99224, 7)


Створення typed-таблиць SQL-кодом

In [28]:
%%sql

SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM olist_customers_raw
UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders_raw
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items_raw
UNION ALL SELECT 'products', COUNT(*) FROM olist_products_raw
UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers_raw
UNION ALL SELECT 'reviews', COUNT(*) FROM olist_order_reviews_raw
ORDER BY table_name;

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-04%2Fhw4_pg
6 rows affected.


,table_name,n_rows
0,customers,99441
1,order_items,112650
2,orders,99441
3,products,32951
4,reviews,99224
5,sellers,3095


In [29]:
%%sql

DROP TABLE IF EXISTS customer_segments CASCADE;
DROP TABLE IF EXISTS seller_score CASCADE;
DROP TABLE IF EXISTS seller_alerts CASCADE;
DROP TABLE IF EXISTS olist_customer_dim CASCADE;
DROP TABLE IF EXISTS olist_order_reviews CASCADE;
DROP TABLE IF EXISTS olist_order_items CASCADE;
DROP TABLE IF EXISTS olist_orders CASCADE;
DROP TABLE IF EXISTS olist_customers CASCADE;
DROP TABLE IF EXISTS olist_products CASCADE;
DROP TABLE IF EXISTS olist_sellers CASCADE;

CREATE TABLE olist_customers (
    customer_id TEXT PRIMARY KEY,
    customer_unique_id TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city TEXT,
    customer_state CHAR(2) NOT NULL
);

CREATE INDEX idx_olist_customers_unique_id
ON olist_customers (customer_unique_id);

CREATE TABLE olist_orders (
    order_id TEXT PRIMARY KEY,
    customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status TEXT NOT NULL,
    order_purchase_timestamp TIMESTAMP NOT NULL,
    order_approved_at TIMESTAMP,
    order_delivered_carrier_date TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id TEXT PRIMARY KEY,
    product_category_name TEXT,
    product_name_length INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city TEXT,
    seller_state CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id INTEGER NOT NULL,
    product_id TEXT REFERENCES olist_products(product_id),
    seller_id TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);

CREATE INDEX idx_olist_order_items_seller_order
ON olist_order_items (seller_id, order_id);

CREATE TABLE olist_order_reviews (
    review_row_id BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    review_id TEXT,
    order_id TEXT NOT NULL REFERENCES olist_orders(order_id),
    review_score SMALLINT CHECK (review_score BETWEEN 1 AND 5),
    review_comment_title TEXT,
    review_comment_message TEXT,
    review_creation_date TIMESTAMP,
    review_answer_timestamp TIMESTAMP
);

CREATE INDEX idx_olist_order_reviews_order_id
ON olist_order_reviews (order_id);

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-04%2Fhw4_pg
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.


""


Перенесення даних з raw у typed

In [32]:
%%sql

INSERT INTO olist_customers (
    customer_id, customer_unique_id,
    customer_zip_code_prefix, customer_city, customer_state
)
SELECT
    customer_id, customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city, customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders (
    order_id, customer_id, order_status,
    order_purchase_timestamp, order_approved_at,
    order_delivered_carrier_date,
    order_delivered_customer_date,
    order_estimated_delivery_date
)
SELECT
    order_id, customer_id, order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products (
    product_id, product_category_name,
    product_name_length, product_description_length,
    product_photos_qty, product_weight_g,
    product_length_cm, product_height_cm, product_width_cm
)
SELECT
    product_id, product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers (
    seller_id, seller_zip_code_prefix,
    seller_city, seller_state
)
SELECT
    seller_id, seller_zip_code_prefix::INTEGER,
    seller_city, seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items (
    order_id, order_item_id, product_id,
    seller_id, shipping_limit_date,
    price, freight_value
)
SELECT
    order_id, order_item_id::INTEGER,
    product_id, seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2),
    freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;

INSERT INTO olist_order_reviews (
    review_id, order_id, review_score,
    review_comment_title, review_comment_message,
    review_creation_date, review_answer_timestamp
)
SELECT
    review_id, order_id,
    review_score::SMALLINT,
    review_comment_title,
    review_comment_message,
    review_creation_date::TIMESTAMP,
    review_answer_timestamp::TIMESTAMP
FROM olist_order_reviews_raw;

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-04%2Fhw4_pg
99441 rows affected.
99441 rows affected.
32951 rows affected.
3095 rows affected.
112650 rows affected.
99224 rows affected.


""


Smoke-test кількості рядків

In [33]:
%%sql

SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM olist_customers
UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL SELECT 'products', COUNT(*) FROM olist_products
UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers
UNION ALL SELECT 'reviews', COUNT(*) FROM olist_order_reviews
ORDER BY table_name;

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-04%2Fhw4_pg
6 rows affected.


,table_name,n_rows
0,customers,99441
1,order_items,112650
2,orders,99441
3,products,32951
4,reviews,99224
5,sellers,3095


In [34]:
%%sql

SELECT
    'customers' AS table_name,
    (SELECT COUNT(*) FROM olist_customers_raw) AS raw_rows,
    (SELECT COUNT(*) FROM olist_customers) AS typed_rows

UNION ALL

SELECT
    'orders',
    (SELECT COUNT(*) FROM olist_orders_raw),
    (SELECT COUNT(*) FROM olist_orders)

UNION ALL

SELECT
    'order_items',
    (SELECT COUNT(*) FROM olist_order_items_raw),
    (SELECT COUNT(*) FROM olist_order_items)

UNION ALL

SELECT
    'products',
    (SELECT COUNT(*) FROM olist_products_raw),
    (SELECT COUNT(*) FROM olist_products)

UNION ALL

SELECT
    'sellers',
    (SELECT COUNT(*) FROM olist_sellers_raw),
    (SELECT COUNT(*) FROM olist_sellers)

UNION ALL

SELECT
    'reviews',
    (SELECT COUNT(*) FROM olist_order_reviews_raw),
    (SELECT COUNT(*) FROM olist_order_reviews);

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-04%2Fhw4_pg
6 rows affected.


,table_name,raw_rows,typed_rows
0,customers,99441,99441
1,orders,99441,99441
2,order_items,112650,112650
3,products,32951,32951
4,sellers,3095,3095
5,reviews,99224,99224


Завдання 2. DML-запити з PostgreSQL-фічами



2.1. INSERT ... RETURNING: створення alert для продавця з найбільшою кількістю запізнень



In [35]:
%%sql

DROP TABLE IF EXISTS seller_alerts CASCADE;

CREATE TABLE seller_alerts (
    alert_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    seller_id TEXT NOT NULL REFERENCES olist_sellers(seller_id),
    alert_type TEXT NOT NULL CHECK (
        alert_type IN ('late_delivery', 'low_rating', 'data_quality')
    ),
    severity SMALLINT NOT NULL CHECK (severity BETWEEN 1 AND 5),
    details TEXT,
    created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
);

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-04%2Fhw4_pg
Done.
Done.


""


In [36]:
%%sql

WITH seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.order_id
    FROM olist_order_items AS oi
),
late_sellers AS (
    SELECT
        so.seller_id,
        COUNT(DISTINCT o.order_id) AS late_orders
    FROM seller_orders AS so
    JOIN olist_orders AS o
        ON o.order_id = so.order_id
    WHERE o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
      AND o.order_delivered_customer_date::DATE >
          o.order_estimated_delivery_date::DATE
    GROUP BY so.seller_id
    ORDER BY late_orders DESC, so.seller_id
    LIMIT 1
)
INSERT INTO seller_alerts (
    seller_id,
    alert_type,
    severity,
    details
)
SELECT
    seller_id,
    'late_delivery',
    4,
    'Seller has ' || late_orders ||
    ' late delivered orders in the loaded dataset'
FROM late_sellers
RETURNING
    alert_id,
    seller_id,
    alert_type,
    severity,
    created_at;

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-04%2Fhw4_pg
1 rows affected.


,alert_id,seller_id,alert_type,severity,created_at
0,1,4a3ca9315b744ce9f8e9374361493884,late_delivery,4,2026-11-01 17:43:55.593128+01:00


In [37]:
%%sql

SELECT
    alert_id,
    seller_id,
    alert_type,
    severity,
    details,
    created_at
FROM seller_alerts
ORDER BY created_at DESC;

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-04%2Fhw4_pg
1 rows affected.


,alert_id,seller_id,alert_type,severity,details,created_at
0,1,4a3ca9315b744ce9f8e9374361493884,late_delivery,4,Seller has 172 late delivered orders in the lo...,2026-11-01 17:43:55.593128+01:00


Запит визначив продавця `4a3ac9315b744ce9f8e9374361493884`, який має найбільшу кількість запізнілих доставок — **172 замовлення**. Для нього створено попередження типу `late_delivery` з рівнем серйозності 4.

Конструкція `RETURNING` дозволила одразу отримати ідентифікатор створеного попередження (`alert_id = 1`), інформацію про продавця та час створення запису. Ці дані можна використовувати як audit-log для відстеження змін у базі даних.

2.2. INSERT ... ON CONFLICT DO UPDATE: idempotent upsert агрегованих seller-score


In [38]:
%%sql

DROP TABLE IF EXISTS seller_score CASCADE;

CREATE TABLE seller_score (
    seller_id TEXT PRIMARY KEY REFERENCES olist_sellers(seller_id),
    avg_rating NUMERIC(3, 2),
    n_reviews INTEGER NOT NULL DEFAULT 0 CHECK (n_reviews >= 0),
    tier TEXT CHECK (tier IN ('gold', 'silver', 'bronze')),
    updated_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
);

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-04%2Fhw4_pg
Done.
Done.


""


Розрахунок seller-score через UPSERT

In [39]:
%%sql

WITH seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.order_id
    FROM olist_order_items AS oi
),
review_per_order AS (
    SELECT
        rv.order_id,
        AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
seller_rating AS (
    SELECT
        so.seller_id,
        AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_rating,
        COUNT(rpo.order_id) AS n_reviews
    FROM seller_orders AS so
    LEFT JOIN review_per_order AS rpo
        ON rpo.order_id = so.order_id
    GROUP BY so.seller_id
)
INSERT INTO seller_score (
    seller_id,
    avg_rating,
    n_reviews,
    tier
)
SELECT
    seller_id,
    avg_rating,
    n_reviews,
    CASE
        WHEN avg_rating IS NULL THEN NULL
        WHEN avg_rating >= 4.5 THEN 'gold'
        WHEN avg_rating >= 3.5 THEN 'silver'
        ELSE 'bronze'
    END AS tier
FROM seller_rating
ON CONFLICT (seller_id) DO UPDATE
SET
    avg_rating = EXCLUDED.avg_rating,
    n_reviews = EXCLUDED.n_reviews,
    tier = EXCLUDED.tier,
    updated_at = NOW()
RETURNING
    seller_id,
    avg_rating,
    n_reviews,
    tier,
    updated_at;

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-04%2Fhw4_pg
3095 rows affected.


,seller_id,avg_rating,n_reviews,tier,updated_at
0,a61cc04793308395a840807104365121,5.00,1,gold,2026-11-01 17:54:40.664359+01:00
1,6d2f2e3b539480db1e0842b3a4e32e6e,3.67,3,silver,2026-11-01 17:54:40.664359+01:00
2,c53bcd3be457a342a97e39e5a9f0be22,3.20,5,bronze,2026-11-01 17:54:40.664359+01:00
3,9591fc341b1bfb7ef561e2968ec6e011,1.00,1,bronze,2026-11-01 17:54:40.664359+01:00
4,f9ec7093df3a7b346b7bcf7864069ca3,4.78,9,gold,2026-11-01 17:54:40.664359+01:00
...,...,...,...,...,...
3090,b2eecf5ea250510da76590ca79d60e5d,5.00,9,gold,2026-11-01 17:54:40.664359+01:00
3091,eae9af4811c294c56795d70e715b7337,5.00,1,gold,2026-11-01 17:54:40.664359+01:00
3092,1c83418358f249c9d6affa52d579db0e,5.00,1,gold,2026-11-01 17:54:40.664359+01:00
3093,845e8036c3f7bf62099352f5ed910240,3.95,21,silver,2026-11-01 17:54:40.664359+01:00


Перевірка результатів

In [40]:
%%sql

SELECT
    tier,
    COUNT(*) AS total_sellers,
    ROUND(AVG(avg_rating), 2) AS avg_rating,
    SUM(n_reviews) AS total_reviews
FROM seller_score
GROUP BY tier
ORDER BY avg_rating DESC NULLS LAST;

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-04%2Fhw4_pg
4 rows affected.


,tier,total_sellers,avg_rating,total_reviews
0,gold,983,4.83,7093
1,silver,1568,4.06,87123
2,bronze,539,2.30,5031
3,NaN,5,None,0


### 2.2. Interpretation — INSERT ... ON CONFLICT DO UPDATE

`SELECT DISTINCT` у `seller_orders` використовується для отримання унікальних пар `seller_id × order_id`. Це запобігає повторному врахуванню одного замовлення, якщо воно містить декілька товарів одного продавця.

Відгуки спочатку агрегуються на рівні `order_id`, оскільки одне замовлення може мати декілька review-записів. Це дозволяє розрахувати середню оцінку замовлення та уникнути штучного збільшення ваги окремих відгуків під час обчислення рейтингу продавця.

`ON CONFLICT (seller_id) DO UPDATE` забезпечує повторюваність запиту без дублювання рядків: якщо продавець уже існує в таблиці, його рейтинг, кількість оцінених замовлень і категорія оновлюються. При повторному запуску також оновлюється `updated_at`.

Продавці без відгуків мають `avg_rating = NULL`, `n_reviews = 0` і `tier = NULL`, оскільки для них немає даних для розрахунку рейтингу.

**Результати аналізу:**

- **Gold:** 983 продавці, середній рейтинг 4.83, 7 093 оцінені замовлення.
- **Silver:** 1 568 продавців, середній рейтинг 4.06, 87 123 оцінені замовлення.
- **Bronze:** 539 продавців, середній рейтинг 2.30, 5 031 оцінене замовлення.
- **Без рейтингу:** 5 продавців, для яких відсутні оцінені замовлення.

Найбільша кількість продавців належить до категорії `silver` — приблизно 50,7% від загальної кількості. Це свідчить про переважання продавців із середнім рейтингом від 3.5 до 4.49.

**Обмеження датасету:** в Olist відгуки прив'язані до замовлення, а не до конкретного продавця. Якщо замовлення містить товари кількох продавців, його оцінка враховується для кожного з них. Це прийнятне навчальне спрощення, однак воно може впливати на точність індивідуального рейтингу продавців.

2.3. UPDATE ... FROM: оновлення прапорця запізнення в olist_orders



In [41]:

%%sql

ALTER TABLE olist_orders
    ADD COLUMN IF NOT EXISTS is_late BOOLEAN NOT NULL DEFAULT FALSE;

WITH delivery_flags AS (
    SELECT
        order_id,
        (
            order_delivered_customer_date IS NOT NULL
            AND order_estimated_delivery_date IS NOT NULL
            AND order_delivered_customer_date::DATE > order_estimated_delivery_date::DATE
        ) AS is_late_calc
    FROM olist_orders
)
UPDATE olist_orders AS o
SET is_late = f.is_late_calc
FROM delivery_flags AS f
WHERE f.order_id = o.order_id
RETURNING o.order_id, o.customer_id, o.is_late;


 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-04%2Fhw4_pg
Done.
99441 rows affected.


,order_id,customer_id,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,False
...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,False
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,False
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,False
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,False


In [42]:

%%sql

SELECT
    is_late,
    COUNT(*) AS total_orders,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS percentage
FROM olist_orders
GROUP BY is_late
ORDER BY is_late DESC;


 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-04%2Fhw4_pg
2 rows affected.


,is_late,total_orders,percentage
0,True,6535,6.57
1,False,92906,93.43


### 2.3. UPDATE ... FROM — оновлення прапорця запізнення

У цьому завданні до таблиці `olist_orders` додано колонку `is_late`, яка визначає, чи було замовлення доставлено пізніше запланованої дати.

За допомогою `WITH delivery_flags` для кожного замовлення розраховується значення `TRUE` або `FALSE` на основі порівняння фактичної та очікуваної дат доставки.

Конструкція `UPDATE ... FROM` дозволяє оновити значення `is_late` для всіх замовлень, а `RETURNING` — переглянути оновлені записи та використати їх для перевірки результатів.

**Запит є ідемпотентним**, оскільки під час повторного запуску значення `is_late` перераховується відповідно до поточних дат доставки. Це запобігає накопиченню некоректних значень і забезпечує повторюваність результатів.

Замовлення без фактичної або очікуваної дати доставки отримують `FALSE`, тому їх не слід автоматично інтерпретувати як доставлені вчасно.

2.4. DELETE ... RETURNING — контрольоване видалення застарілих alert-ів

In [43]:

%%sql

DELETE FROM seller_alerts
WHERE created_at < NOW() - INTERVAL '30 days'
RETURNING alert_id, seller_id, alert_type, created_at;


 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-04%2Fhw4_pg
0 rows affected.


""


In [44]:

%%sql

SELECT
    COUNT(*) AS total_alerts,
    COUNT(*) FILTER (
        WHERE created_at < NOW() - INTERVAL '30 days'
    ) AS outdated_alerts
FROM seller_alerts;


 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-04%2Fhw4_pg
1 rows affected.


,total_alerts,outdated_alerts
0,1,0


### 2.4. DELETE ... RETURNING — контрольоване видалення застарілих alert-ів

У цьому завданні використано команду `DELETE` для видалення сповіщень із таблиці `seller_alerts`, створених понад 30 днів тому.

Умова `WHERE created_at < NOW() - INTERVAL '30 days'` гарантує, що видаляються лише застарілі записи, а нові сповіщення залишаються в таблиці.

Конструкція `RETURNING` дозволяє одразу отримати інформацію про видалені записи (`alert_id`, `seller_id`, `alert_type`, `created_at`). Ці дані можна використати для аудиту операції.

**Запит повертає 0 рядків, це допустимий результат**, оскільки в таблиці лише нещодавно створені сповіщення, які ще не досягли 30-денного терміну.

Запит безпечний для повторного виконання: після першого видалення застарілих записів наступний запуск не змінить таблицю, якщо нові записи не стали застарілими.

Завдання 3. CREATE TABLE customer_segments



In [45]:

%%sql

DROP TABLE IF EXISTS customer_segments CASCADE;
DROP TABLE IF EXISTS olist_customer_dim CASCADE;

CREATE TABLE olist_customer_dim (
    customer_unique_id TEXT PRIMARY KEY,
    latest_customer_id TEXT REFERENCES olist_customers(customer_id),
    customer_state     CHAR(2),
    customer_city      TEXT
);

INSERT INTO olist_customer_dim (
    customer_unique_id,
    latest_customer_id,
    customer_state,
    customer_city
)
SELECT DISTINCT ON (c.customer_unique_id)
    c.customer_unique_id,
    c.customer_id AS latest_customer_id,
    c.customer_state,
    c.customer_city
FROM olist_customers AS c
LEFT JOIN olist_orders AS o
    ON o.customer_id = c.customer_id
ORDER BY
    c.customer_unique_id,
    o.order_purchase_timestamp DESC NULLS LAST,
    c.customer_id;


 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-04%2Fhw4_pg
Done.
Done.
Done.
96096 rows affected.


""


Customer_segments з production-подібним DDL: identity PK, FK, UNIQUE, CHECK, NOT NULL, DEFAULT і idempotent-наповнення через INSERT ... SELECT ... ON CONFLICT

In [46]:

%%sql

CREATE TABLE customer_segments (
    segment_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    customer_unique_id TEXT NOT NULL,

    segment_name TEXT NOT NULL
        CHECK (segment_name IN ('VIP', 'regular', 'new', 'inactive')),

    rfm_score SMALLINT NOT NULL
        CHECK (rfm_score BETWEEN 1 AND 5),

    recency_days INTEGER
        CHECK (recency_days IS NULL OR recency_days >= 0),

    monetary_value NUMERIC(12, 2) NOT NULL DEFAULT 0
        CHECK (monetary_value >= 0),

    n_orders INTEGER NOT NULL DEFAULT 0
        CHECK (n_orders >= 0),

    assigned_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),

    UNIQUE (customer_unique_id)
);

ALTER TABLE customer_segments
    ADD CONSTRAINT fk_customer_segments_customer_unique
    FOREIGN KEY (customer_unique_id)
    REFERENCES olist_customer_dim(customer_unique_id)
    ON DELETE CASCADE;


 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-04%2Fhw4_pg
Done.
Done.


""


In [47]:

%%sql

SELECT
    MAX(order_purchase_timestamp)::DATE + 1 AS reference_date
FROM olist_orders;


 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-04%2Fhw4_pg
1 rows affected.


,reference_date
0,2018-10-18


In [48]:

%%sql

WITH reference_date AS (
    SELECT
        (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),

order_totals AS (
    SELECT
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(
            SUM(oi.price + oi.freight_value), 0
        )::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp
),

customer_stats AS (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT ot.order_id) AS n_orders,
        COALESCE(
            SUM(ot.order_total), 0
        )::NUMERIC(12, 2) AS monetary_value,
        MAX(ot.order_date) AS last_order_date
    FROM olist_customer_dim AS cd
    JOIN olist_customers AS c
        ON c.customer_unique_id = cd.customer_unique_id
    LEFT JOIN order_totals AS ot
        ON ot.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
),

segmented AS (
    SELECT
        cs.customer_unique_id,
        cs.n_orders,
        cs.monetary_value,

        CASE
            WHEN cs.last_order_date IS NULL THEN NULL
            ELSE (rd.as_of_date - cs.last_order_date)::INTEGER
        END AS recency_days,

        CASE
            WHEN cs.monetary_value >= 1000 THEN 'VIP'

            WHEN cs.last_order_date IS NOT NULL
                 AND (rd.as_of_date - cs.last_order_date)::INTEGER <= 90
                 AND cs.n_orders = 1
            THEN 'new'

            WHEN cs.n_orders >= 2
                 OR cs.monetary_value >= 100
            THEN 'regular'

            ELSE 'inactive'
        END AS segment_name,

        CASE
            WHEN cs.monetary_value >= 1000 THEN 5
            WHEN cs.monetary_value >= 500 THEN 4
            WHEN cs.n_orders >= 2 THEN 3
            WHEN cs.n_orders = 1 THEN 2
            ELSE 1
        END::SMALLINT AS rfm_score

    FROM customer_stats AS cs
    CROSS JOIN reference_date AS rd
)

INSERT INTO customer_segments (
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders
)

SELECT
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders
FROM segmented

ON CONFLICT (customer_unique_id) DO UPDATE
SET
    segment_name = EXCLUDED.segment_name,
    rfm_score = EXCLUDED.rfm_score,
    recency_days = EXCLUDED.recency_days,
    monetary_value = EXCLUDED.monetary_value,
    n_orders = EXCLUDED.n_orders,
    assigned_at = NOW()

RETURNING
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders;


 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-04%2Fhw4_pg
96096 rows affected.


,customer_unique_id,segment_name,rfm_score,recency_days,monetary_value,n_orders
0,0000366f3b9a7992bf8c76cfdf3221e2,regular,2,161,141.90,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,inactive,2,164,27.19,1
2,0000f46a3911fa3c0805444483337064,inactive,2,587,86.22,1
3,0000f6ccb0745a6a4b88665a16c9f078,inactive,2,371,43.62,1
4,0004aac84e0df4da2b147fca70cf8255,regular,2,338,196.89,1
...,...,...,...,...,...,...
96091,fffcf5a5ff07b0908bd4e2dbc735a684,VIP,5,497,2067.42,1
96092,fffea47cd6d3cc0a88bd621562a9d061,inactive,2,312,84.58,1
96093,ffff371b4d645b6ecea244b27531430a,regular,2,618,112.46,1
96094,ffff5962728ec6157033ef9805bacc48,regular,2,169,133.69,1


розподіл покупців за сегментами

In [49]:

%%sql

SELECT
    segment_name,
    COUNT(*) AS total_customers,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS percentage,
    ROUND(AVG(monetary_value), 2) AS avg_monetary,
    ROUND(AVG(n_orders), 2) AS avg_orders,
    ROUND(AVG(recency_days), 2) AS avg_recency
FROM customer_segments
GROUP BY segment_name
ORDER BY total_customers DESC;


 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-04%2Fhw4_pg
4 rows affected.


,segment_name,total_customers,percentage,avg_monetary,avg_orders,avg_recency
0,regular,45982,47.85,224.57,1.07,307.46
1,inactive,40032,41.66,59.28,1.00,316.14
2,new,8891,9.25,140.39,1.00,72.36
3,VIP,1191,1.24,1591.81,1.11,290.16


Перевіряємо constraints

In [50]:

%%sql

SELECT
    conname,
    contype,
    pg_get_constraintdef(oid) AS definition
FROM pg_constraint
WHERE conrelid = 'customer_segments'::regclass
ORDER BY conname;


 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-04%2Fhw4_pg
8 rows affected.


,conname,contype,definition
0,customer_segments_customer_unique_id_key,u,UNIQUE (customer_unique_id)
1,customer_segments_monetary_value_check,c,CHECK ((monetary_value >= (0)::numeric))
2,customer_segments_n_orders_check,c,CHECK ((n_orders >= 0))
3,customer_segments_pkey,p,PRIMARY KEY (segment_id)
4,customer_segments_recency_days_check,c,CHECK (((recency_days IS NULL) OR (recency_day...
5,customer_segments_rfm_score_check,c,CHECK (((rfm_score >= 1) AND (rfm_score <= 5)))
6,customer_segments_segment_name_check,c,"CHECK ((segment_name = ANY (ARRAY['VIP'::text,..."
7,fk_customer_segments_customer_unique,f,FOREIGN KEY (customer_unique_id) REFERENCES ol...


### Пояснення до завдання 3

1. **Чому використовується `customer_unique_id`?**  
   `customer_unique_id` ідентифікує реального покупця, навіть якщо він зробив декілька замовлень. Це дозволяє правильно розрахувати кількість покупок (Frequency) та загальні витрати (Monetary).

2. **Навіщо потрібна `olist_customer_dim`?**  
   Таблиця містить один запис для кожного `customer_unique_id` і забезпечує унікальність покупців. Завдяки цьому її можна використовувати як довідкову таблицю для `FOREIGN KEY` у `customer_segments`, гарантуючи цілісність даних.

3. **Чому `reference_date` береться з датасету?**  
   Olist — історичний датасет. Використання `CURRENT_DATE` призводило б до постійного збільшення Recency. Дата останнього замовлення плюс один день забезпечує стабільні та відтворювані результати.

4. **Як працює `ON CONFLICT`?**  
   `ON CONFLICT (customer_unique_id) DO UPDATE` дозволяє повторно запускати сегментацію без створення дублікатів. Якщо покупець уже є в таблиці, його показники оновлюються, а якщо немає — додається новий запис. Таким чином, повторний запуск на тих самих даних зберігає однакові результати сегментації, хоча `assigned_at` оновлюється.

Завдання 4. JOIN-запити з business-interpretation

Приклад 4.1. INNER JOIN: топ категорій за revenue і seller-state



Які категорії товарів і штати продавців генерують найбільший revenue серед доставлених замовлень?

In [51]:

%%sql

SELECT
    p.product_category_name,
    s.seller_state,
    COUNT(DISTINCT oi.order_id) AS n_orders,
    SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS revenue
FROM olist_order_items AS oi

INNER JOIN olist_orders AS o
    ON o.order_id = oi.order_id

INNER JOIN olist_products AS p
    ON p.product_id = oi.product_id

INNER JOIN olist_sellers AS s
    ON s.seller_id = oi.seller_id

WHERE o.order_status = 'delivered'

GROUP BY
    p.product_category_name,
    s.seller_state

ORDER BY revenue DESC NULLS LAST

LIMIT 10;


 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-04%2Fhw4_pg
10 rows affected.


,product_category_name,seller_state,n_orders,revenue
0,cama_mesa_banho,SP,8214,1070048.14
1,relogios_presentes,SP,4494,1016168.80
2,beleza_saude,SP,5684,790967.66
3,esporte_lazer,SP,4772,680165.54
4,moveis_decoracao,SP,4597,605650.63
5,cool_stuff,SP,2621,484196.53
6,utilidades_domesticas,SP,4021,471810.66
7,automotivo,SP,3018,438833.22
8,informatica_acessorios,SP,2783,388506.42
9,bebes,SP,1981,339129.74


### 4.1. INNER JOIN — топ категорій за revenue і seller-state

**Business-question:** Які категорії товарів і штати продавців генерують найбільший revenue серед доставлених замовлень?

**Interpretation:**

За результатами аналізу всі 10 комбінацій із найбільшим revenue належать продавцям зі штату **São Paulo (SP)**, що свідчить про його домінування серед лідерів продажів у датасеті Olist.

Найбільший revenue генерують такі категорії:

- **cama_mesa_banho** (товари для спальні, столу та ванної) — 1 070 048,14 BRL, 8 214 замовлень.
- **relogios_presentes** (годинники та подарунки) — 1 016 168,80 BRL, 4 494 замовлення.
- **beleza_saude** (краса та здоров'я) — 790 967,66 BRL, 5 684 замовлення.

Категорія `cama_mesa_banho` лідирує як за revenue, так і за кількістю замовлень серед представлених комбінацій. Водночас `relogios_presentes` генерує майже такий самий revenue за значно меншої кількості замовлень, що може свідчити про вищу середню вартість продажів.

**Business insight:** São Paulo є ключовим штатом продавців серед топових комбінацій. Для маркетингових кампаній доцільно пріоритизувати категорії `cama_mesa_banho`, `relogios_presentes` і `beleza_saude`, оскільки вони демонструють найбільший обсяг продажів у цьому рейтингу.

4.2 LEFT JOIN + COALESCE — Customer-level Feature Engineering
Business-question: Які характеристики (features) можна побудувати для кожного реального покупця на рівні customer_unique_id?

In [52]:

%%sql

WITH order_totals AS (
    SELECT
        o.order_id,
        c.customer_unique_id,
        SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        c.customer_unique_id
),

review_per_order AS (
    SELECT
        rv.order_id,
        AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),

customer_stats AS (
    SELECT
        ot.customer_unique_id,
        COUNT(DISTINCT ot.order_id) AS n_orders,
        COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS total_spend,
        AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_review_score
    FROM order_totals AS ot
    LEFT JOIN review_per_order AS rpo
        ON rpo.order_id = ot.order_id
    GROUP BY ot.customer_unique_id
)

SELECT
    cd.customer_unique_id,
    cd.customer_state,
    COALESCE(cs.n_orders, 0) AS n_orders,
    COALESCE(cs.total_spend, 0)::NUMERIC(12, 2) AS total_spend,
    COALESCE(cs.avg_review_score, 0)::NUMERIC(3, 2) AS avg_review_score
FROM olist_customer_dim AS cd
LEFT JOIN customer_stats AS cs
    ON cs.customer_unique_id = cd.customer_unique_id
ORDER BY total_spend DESC NULLS LAST
LIMIT 20;


 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-04%2Fhw4_pg
20 rows affected.


,customer_unique_id,customer_state,n_orders,total_spend,avg_review_score
0,0a0a92112bd4c708ca5fde585afaa872,RJ,1,13664.08,1.00
1,da122df9eeddfedc1dc1f5349a1a690c,RJ,2,7571.63,5.00
2,763c8b1c9c68a0229c42c9fc6f662b93,ES,1,7274.88,1.00
3,dc4802a71eae9be1dd28f5d788ceb526,MS,1,6929.31,5.00
4,459bef486812aa25204be022145caa62,ES,1,6922.21,0.00
5,ff4159b92c40ebe40454e3e6a7c35ed6,SP,1,6726.66,5.00
6,4007669dec559734d6f53e029e360987,MG,1,6081.54,1.00
7,5d0a2980b292d049061542014e8960bf,GO,1,4809.44,1.00
8,eebb5dda148d3893cdaf5b5ca3040ccb,SP,1,4764.34,4.00
9,48e1ac109decbb87765a3eade6854098,PB,1,4681.78,5.00


### 4.2. LEFT JOIN + COALESCE — Customer-level Features

**Business-question:** Які customer-level features можна побудувати для реального покупця на рівні `customer_unique_id`?

**Interpretation:**

За допомогою `LEFT JOIN` та `COALESCE` сформовано характеристики покупців: кількість замовлень (`n_orders`), загальні витрати (`total_spend`), середню оцінку відгуків (`avg_review_score`) та штат проживання (`customer_state`).

Аналіз 20 покупців із найбільшими витратами показав:

- Найбільші витрати має покупець зі штату **RJ** — 13 664,08 BRL за одне замовлення, але його оцінка становить лише **1/5**.
- Другий покупець також зі штату **RJ** витратив 7 571,63 BRL за два замовлення із середньою оцінкою **5/5**.
- Більшість покупців у топ-20 здійснили лише одне замовлення, навіть за високої загальної суми витрат.
- Високі витрати не завжди супроводжуються високою задоволеністю: серед найбільших покупців є оцінки як 1/5, так і 5/5.

**Важливість `review_per_order`:**

CTE `review_per_order` попередньо агрегує відгуки до одного рядка на `order_id`. Це захищає від множення `review_score` через декілька товарних позицій одного замовлення під час JOIN. Аналогічно `order_totals` окремо обчислює вартість кожного замовлення. Такий підхід запобігає дублюванню даних і спотворенню агрегованих показників.

`LEFT JOIN` зберігає всіх покупців із `olist_customer_dim`, а `COALESCE` замінює відсутні числові значення на 0. Для `avg_review_score` нуль означає відсутність відгуку, а не фактичну оцінку.

**Business insight:** Customer-level features дозволяють виявляти цінних покупців, аналізувати повторні покупки та оцінювати задоволеність клієнтів. Особливу увагу варто приділити покупцям із високими витратами та низькими оцінками, оскільки вони можуть потребувати додаткової роботи з клієнтським досвідом.

Приклад 4.3. FULL OUTER JOIN: які штати присутні серед клієнтів і продавців

Business-question: які штати представлені тільки серед клієнтів, тільки серед продавців або в обох групах?

In [53]:

%%sql

WITH customer_states AS (
    SELECT
        customer_state AS state,
        COUNT(*) AS n_customers
    FROM olist_customers
    GROUP BY customer_state
),

seller_states AS (
    SELECT
        seller_state AS state,
        COUNT(*) AS n_sellers
    FROM olist_sellers
    GROUP BY seller_state
)

SELECT
    COALESCE(c.state, s.state) AS state,
    c.n_customers,
    s.n_sellers,

    CASE
        WHEN c.state IS NOT NULL
             AND s.state IS NOT NULL
        THEN 'both'

        WHEN c.state IS NOT NULL
        THEN 'customers_only'

        ELSE 'sellers_only'
    END AS state_presence

FROM customer_states AS c

FULL OUTER JOIN seller_states AS s
    ON s.state = c.state

ORDER BY state;


 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-04%2Fhw4_pg
27 rows affected.


,state,n_customers,n_sellers,state_presence
0,AC,81,1.0,both
1,AL,413,NaN,customers_only
2,AM,148,1.0,both
3,AP,68,NaN,customers_only
4,BA,3380,19.0,both
5,CE,1336,13.0,both
6,DF,2140,30.0,both
7,ES,2033,23.0,both
8,GO,2020,40.0,both
9,MA,747,1.0,both


### 4.3. FULL OUTER JOIN — порівняння штатів клієнтів і продавців

**Business-question:** Які штати представлені тільки серед клієнтів, тільки серед продавців або в обох групах?

**Interpretation:**

За допомогою `FULL OUTER JOIN` порівняно географічне розташування клієнтів і продавців у датасеті Olist.

Аналіз показав:

- **23 штати** представлені одночасно серед клієнтів і продавців (`both`).
- **4 штати** мають клієнтів, але не мають зареєстрованих продавців (`customers_only`): AL (413 клієнтів), AP (68), RR (46) і TO (280).
- Штатів, представлених лише серед продавців (`sellers_only`), не виявлено.
- **São Paulo (SP)** має найбільшу кількість клієнтських записів (41 746) і продавців (1 849).
- Інші великі ринки: Rio de Janeiro (RJ) — 12 852 клієнтські записи та 171 продавець; Minas Gerais (MG) — 11 635 клієнтських записів і 244 продавці.

**Business insight:**

São Paulo є найбільшим ринком за кількістю клієнтських записів і продавців, що робить його важливим регіоном для маркетингових і комерційних активностей.

Штати AL, AP, RR і TO можуть бути потенційними напрямками для залучення нових продавців, оскільки попит із боку клієнтів уже присутній, але місцевих продавців у датасеті немає.

Водночас `n_customers` відображає кількість записів у `olist_customers`, а не кількість унікальних покупців. Тому результати характеризують географічне покриття маркетплейсу, а не точний розмір його клієнтської бази.

Приклад 4.4. SELF JOIN: пари продавців з одного штату, але з різних міст

Business-question: які seller-пари можуть бути кандидатами для регіонального порівняння в межах одного штату?

In [54]:

%%sql

SELECT
    s1.seller_id AS seller_a,
    s2.seller_id AS seller_b,
    s1.seller_state,
    s1.seller_city AS city_a,
    s2.seller_city AS city_b
FROM olist_sellers AS s1
JOIN olist_sellers AS s2
    ON s1.seller_state = s2.seller_state
   AND s1.seller_id < s2.seller_id
   AND s1.seller_city IS DISTINCT FROM s2.seller_city
ORDER BY s1.seller_state, seller_a, seller_b
LIMIT 20;


 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-04%2Fhw4_pg
20 rows affected.


,seller_a,seller_b,seller_state,city_a,city_b
0,1444c08e64d55fb3c25f0f09c07ffcf2,2b402d5dc42554061f8ea98d1916f148,BA,lauro de freitas,irece
1,1444c08e64d55fb3c25f0f09c07ffcf2,398cb257329ef7af7f1943a8974a3cbc,BA,lauro de freitas,salvador
2,1444c08e64d55fb3c25f0f09c07ffcf2,4221a7df464f1fe2955934e30ff3a5a1,BA,lauro de freitas,bahia
3,1444c08e64d55fb3c25f0f09c07ffcf2,43753b27d77860f1654aa72e251a7878,BA,lauro de freitas,barro alto
4,1444c08e64d55fb3c25f0f09c07ffcf2,4aba391bc3b88717ce08eb11e44937b2,BA,lauro de freitas,arraial d'ajuda (porto seguro)
5,1444c08e64d55fb3c25f0f09c07ffcf2,4fb41dff7c50136976d1a5cf004a42e2,BA,lauro de freitas,feira de santana
6,1444c08e64d55fb3c25f0f09c07ffcf2,651530bf5c607240ccdd89a30c9c9712,BA,lauro de freitas,ipira
7,1444c08e64d55fb3c25f0f09c07ffcf2,659e8466eb3ff1b0e8740d74fb7bbedd,BA,lauro de freitas,eunapolis
8,1444c08e64d55fb3c25f0f09c07ffcf2,75d34ebb1bd0bd7dde40dd507b8169c3,BA,lauro de freitas,salvador
9,1444c08e64d55fb3c25f0f09c07ffcf2,a3dd39f583bc80bd8c5901c95878921e,BA,lauro de freitas,salvador


### 4.4. SELF JOIN — пари продавців з одного штату

**Business-question:** Які пари продавців можуть бути кандидатами для регіонального порівняння в межах одного штату?

**Interpretation:**

За допомогою `SELF JOIN` знайдено пари продавців, які зареєстровані в одному штаті, але в різних містах.

Усі 20 пар у результаті запиту належать до штату **Bahia (BA)**. Серед представлених міст — Lauro de Freitas, Irecê, Salvador, Feira de Santana, Eunápolis, Ilhéus та Porto Seguro.

Перші 17 пар містять одного продавця з Lauro de Freitas, якого порівнюють із продавцями з інших міст штату Bahia. Наступні три пари містять продавця з Irecê.

Умова `s1.seller_id < s2.seller_id` виключає повторення пар у зворотному порядку та порівняння продавця із самим собою. Оператор `IS DISTINCT FROM` дозволяє порівнювати значення міст з урахуванням `NULL`.

**Business insight:**

Отримані пари можуть використовуватися для регіонального порівняння продавців за кількістю замовлень, revenue, рейтингом і швидкістю доставки.

Такий аналіз допоможе визначити, чи відрізняється ефективність продавців залежно від міста в межах одного штату.

**Обмеження:** Запит показує лише перші 20 пар після сортування, тому результати не відображають повного географічного розподілу продавців. Також він не дозволяє зробити висновок про їхню фактичну ефективність без додаткових показників.

4.5. Anti-join — замовлення без відгуку
Business-question: Які замовлення не мають відгуків від покупців і скільки таких замовлень у датасеті Olist?

In [55]:

%%sql

SELECT
    o.order_id,
    o.customer_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_delivered_customer_date
FROM olist_orders AS o

LEFT JOIN olist_order_reviews AS r
    ON o.order_id = r.order_id

WHERE r.order_id IS NULL

ORDER BY o.order_purchase_timestamp DESC
LIMIT 20;


 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-04%2Fhw4_pg
20 rows affected.


,order_id,customer_id,order_status,order_purchase_timestamp,order_delivered_customer_date
0,afb61112fb99b07fe17e591c68c0c84c,36b182c39933d55e307096ccc3fd2970,canceled,2018-09-06 10:48:12,NaT
1,e91a3bc14b8d125d82f0334371c39926,befe184e4eb181470e6063a1a368a093,delivered,2018-08-27 17:01:43,2018-08-29 20:04:52
2,37e5725572d3cbd7aad103b730f9232a,3dd789d5301761182d28eb73ce1558ed,delivered,2018-08-20 09:18:49,2018-08-29 22:58:53
3,aa6e3cc7cf13ef1dc599d1243daee6f6,af18612bfb6a3f68257c693f72848f28,delivered,2018-08-18 23:54:05,2018-08-27 18:40:41
4,4ac05edf248d6206469e17c32e52c9ac,741f00aca12b97beb2db34f629655c30,delivered,2018-08-17 22:35:51,2018-08-25 12:07:33
5,fea523ebb7e205d346f445544e0a18fd,fbf15dd7058613351fcd0f821a222933,delivered,2018-08-17 12:46:57,2018-08-24 22:12:54
6,a46e7e8e02915adecb392ad065121a51,0ef4273145d74465c180b6a9fbb9799b,delivered,2018-08-17 10:59:18,2018-08-29 20:03:26
7,bcbd3afc999b3fa9dbd61efb370239dd,87394943aaa0bdfb58be8462b549eb95,delivered,2018-08-16 12:31:47,2018-08-23 04:13:29
8,b94d017913e0b697201de9ec22c23f46,cf48656f8158483ef39e8c31b49d7033,delivered,2018-08-16 10:11:23,2018-08-20 19:54:35
9,ff5b7f440481674b38554434638beebf,e6ab213cb124524887ac41b8eaca9653,delivered,2018-08-15 20:24:56,2018-08-30 14:18:52


Завдання 5. GROUP BY, HAVING і агрегати

Виконайте щонайменше 3 агреговані запити з осмисленими бізнес-метриками. У мінімум 2 із 3 запитів використайте HAVING. У щонайменше одному запиті використайте STRING_AGG, ARRAY_AGG або JSONB_AGG.

5.1. Топ категорій за revenue та середнім чеком

Business-question: Які категорії товарів генерують найбільший revenue серед доставлених замовлень із достатнім обсягом продажів?

In [56]:

%%sql

SELECT
    p.product_category_name,
    COUNT(DISTINCT o.order_id) AS n_orders,
    COUNT(*) AS n_items,
    ROUND(SUM(oi.price + oi.freight_value), 2) AS revenue,
    ROUND(
        SUM(oi.price + oi.freight_value)
        / NULLIF(COUNT(DISTINCT o.order_id), 0),
        2
    ) AS avg_revenue_per_order
FROM olist_order_items AS oi
JOIN olist_orders AS o
    ON o.order_id = oi.order_id
JOIN olist_products AS p
    ON p.product_id = oi.product_id
WHERE o.order_status = 'delivered'
  AND p.product_category_name IS NOT NULL
GROUP BY p.product_category_name
HAVING COUNT(DISTINCT o.order_id) >= 100
ORDER BY revenue DESC
LIMIT 10;


 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-04%2Fhw4_pg
10 rows affected.


,product_category_name,n_orders,n_items,revenue,avg_revenue_per_order
0,beleza_saude,8647,9465,1412089.53,163.30
1,relogios_presentes,5495,5859,1264333.12,230.09
2,cama_mesa_banho,9272,10953,1225209.26,132.14
3,esporte_lazer,7530,8431,1118256.91,148.51
4,informatica_acessorios,6530,7644,1032723.77,158.15
5,moveis_decoracao,6307,8160,880329.92,139.58
6,utilidades_domesticas,5743,6795,758392.25,132.06
7,cool_stuff,3559,3718,691680.89,194.35
8,automotivo,3810,4140,669454.75,175.71
9,ferramentas_jardim,3448,4268,567145.68,164.49


5.2. Ефективність продавців: Revenue та рейтинг

Business-question: Які продавці генерують найбільший revenue та який у них середній рейтинг?

In [57]:

%%sql

WITH seller_sales AS (
    SELECT
        oi.seller_id,
        COUNT(DISTINCT oi.order_id) AS n_orders,
        SUM(oi.price + oi.freight_value) AS revenue
    FROM olist_order_items AS oi
    JOIN olist_orders AS o
        ON o.order_id = oi.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY oi.seller_id
    HAVING COUNT(DISTINCT oi.order_id) >= 50
)

SELECT
    s.seller_state,
    COUNT(*) AS n_sellers,
    SUM(ss.n_orders) AS seller_order_count,
    ROUND(SUM(ss.revenue), 2) AS revenue,
    ROUND(AVG(sc.avg_rating), 2) AS avg_seller_rating
FROM seller_sales AS ss
JOIN olist_sellers AS s
    ON s.seller_id = ss.seller_id
LEFT JOIN seller_score AS sc
    ON sc.seller_id = ss.seller_id
GROUP BY s.seller_state
HAVING COUNT(*) >= 5
ORDER BY revenue DESC
LIMIT 10;


 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-04%2Fhw4_pg
6 rows affected.


,seller_state,n_sellers,seller_order_count,revenue,avg_seller_rating
0,SP,286,55071,7261753.71,4.09
1,PR,38,4805,849843.17,4.08
2,MG,38,5563,737188.97,4.16
3,RJ,16,2789,564685.11,4.16
4,SC,23,2521,467331.91,4.11
5,RS,11,1189,226207.63,4.29


5.3. STRING_AGG — Географія покупців за штатами

Business-question: У яких штатах зосереджено найбільше покупців та які міста формують їхню основну клієнтську базу?

In [58]:

%%sql

WITH city_stats AS (
    SELECT
        customer_state,
        customer_city,
        COUNT(*) AS n_customers
    FROM olist_customer_dim
    WHERE customer_state IS NOT NULL
      AND customer_city IS NOT NULL
    GROUP BY customer_state, customer_city
),

ranked_cities AS (
    SELECT
        customer_state,
        customer_city,
        n_customers,
        ROW_NUMBER() OVER (
            PARTITION BY customer_state
            ORDER BY n_customers DESC, customer_city
        ) AS city_rank
    FROM city_stats
)

SELECT
    customer_state,
    SUM(n_customers) AS total_customers,
    COUNT(*) AS n_cities,
    STRING_AGG(
        customer_city || ' (' || n_customers || ')',
        ', ' ORDER BY n_customers DESC, customer_city
    ) FILTER (WHERE city_rank <= 3) AS top_3_cities
FROM ranked_cities
GROUP BY customer_state
ORDER BY total_customers DESC
LIMIT 10;


 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-04%2Fhw4_pg
10 rows affected.


,customer_state,total_customers,n_cities,top_3_cities
0,SP,40296,629,"sao paulo (14972), campinas (1396), guarulhos ..."
1,RJ,12381,149,"rio de janeiro (6612), niteroi (810), nova igu..."
2,MG,11252,745,"belo horizonte (2669), contagem (417), juiz de..."
3,RS,5276,379,"porto alegre (1326), caxias do sul (213), cano..."
4,PR,4879,364,"curitiba (1462), londrina (298), maringa (259)"
5,SC,3531,240,"florianopolis (545), joinville (260), blumenau..."
6,BA,3275,353,"salvador (1207), feira de santana (176), vitor..."
7,DF,2072,6,"brasilia (2065), taguatinga (3), ceilandia (1)"
8,ES,1964,95,"vitoria (366), vila velha (325), serra (278)"
9,GO,1950,177,"goiania (670), anapolis (110), aparecida de go..."


### Різниця між WHERE і HAVING

`WHERE` та `HAVING` використовуються для фільтрації даних, але на різних етапах виконання SQL-запиту.

- **`WHERE`** фільтрує окремі рядки **до агрегації** (`GROUP BY`).
- **`HAVING`** фільтрує сформовані групи **після агрегації**.

**Приклад із завдання 5.1 — аналіз revenue за категоріями товарів:**

```sql
WHERE o.order_status = 'delivered'
  AND p.product_category_name IS NOT NULL

GROUP BY p.product_category_name

HAVING COUNT(DISTINCT o.order_id) >= 100
```

**Пояснення:**

1. `WHERE` залишає лише доставлені замовлення з відомою категорією товару.
2. `GROUP BY` об'єднує відібрані записи за категоріями та дозволяє розрахувати агреговані показники.
3. `HAVING` залишає лише категорії, які мають щонайменше 100 унікальних замовлень.

**Висновок:** `WHERE` визначає, які записи братимуть участь у розрахунках, а `HAVING` — які групи потраплять до фінального результату. Це дозволяє аналізувати лише релевантні замовлення та категорії з достатнім обсягом продажів.

Завдання 6. Set operations, підзапит/window і Reflection

Виконайте щонайменше один set-operation запит. Можна використати UNION, UNION ALL, INTERSECT або EXCEPT, але поясніть, чому обрали саме цей оператор.

6.1. Set Operation — EXCEPT

Business-question: У яких штатах Бразилії є покупці, але немає зареєстрованих продавців Olist?

In [59]:

%%sql

SELECT customer_state AS state
FROM olist_customers

EXCEPT

SELECT seller_state AS state
FROM olist_sellers

ORDER BY state;


 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-04%2Fhw4_pg
4 rows affected.


,state
0,AL
1,AP
2,RR
3,TO


Чому обрали саме EXCEPT?

На відміну від UNION, який об'єднує значення двох наборів, або INTERSECT, який знаходить спільні значення, EXCEPT дозволяє знайти саме відмінність між наборами.

Це відповідає бізнес-питанню: знайти штати, у яких є клієнти, але відсутні продавці.

6.2. Window Function — ROW_NUMBER() OVER (...)

Business-question: Хто входить до топ-3 продавців за revenue в кожному штаті?

In [60]:

%%sql

WITH seller_revenue AS (
    SELECT
        s.seller_id,
        s.seller_state,
        COUNT(DISTINCT oi.order_id) AS n_orders,
        SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS revenue
    FROM olist_sellers AS s
    JOIN olist_order_items AS oi
        ON oi.seller_id = s.seller_id
    JOIN olist_orders AS o
        ON o.order_id = oi.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY
        s.seller_id,
        s.seller_state
),

ranked_sellers AS (
    SELECT
        seller_id,
        seller_state,
        n_orders,
        revenue,
        ROW_NUMBER() OVER (
            PARTITION BY seller_state
            ORDER BY revenue DESC, seller_id
        ) AS seller_rank
    FROM seller_revenue
)

SELECT
    seller_state,
    seller_rank,
    seller_id,
    n_orders,
    revenue
FROM ranked_sellers
WHERE seller_rank <= 3
ORDER BY seller_state, seller_rank;


 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-04%2Fhw4_pg
56 rows affected.


,seller_state,seller_rank,seller_id,n_orders,revenue
0,AM,1,327b89b872c14d1c0be7235ef4871685,3,1258.80
1,BA,1,53243585a1d6dc2643021fd1853d8905,348,230797.02
2,BA,2,c72de06d72748d1a0dfb2125be43ba63,21,18733.18
3,BA,3,75d34ebb1bd0bd7dde40dd507b8169c3,57,13969.77
4,CE,1,bbf9ad41dca6603e614efcdad7aab8c4,14,8622.26
5,CE,2,dbdd0ec73a4817971d962698f2fea022,16,8174.80
6,CE,3,8d79c8a04e42d722a75097ce5cbcf2ef,26,3347.25
7,DF,1,44073f8b7e41514de3b7815dd0237f4f,142,21519.49
8,DF,2,f3b80352b986ab4d1057a4b724be19d0,86,11941.36
9,DF,3,5656537e588803a555b8eb41f07a944b,171,11255.55


### Reflection

Під час виконання домашнього завдання найскладнішим для мене був `FULL OUTER JOIN`, оскільки потрібно було правильно порівняти два набори даних і врахувати записи, присутні лише в одній із таблиць.

Найскладніше було сформулювати business-question для `SELF JOIN`, адже потрібно було не просто знайти пари продавців з одного штату, а й пояснити практичну користь такого порівняння для бізнесу.

Найкориснішою PostgreSQL-функціональністю для production-ETL я вважаю `ON CONFLICT`. Вона дозволяє повторно запускати завантаження даних без створення дублікатів, оновлюючи вже наявні записи.

У реальному ML-проєкті я б використовувала window functions для ранжування клієнтів за витратами, визначення останньої покупки, розрахунку накопичувального revenue та аналізу змін поведінки покупців у часі.

Під час роботи з Olist я звернула увагу на проблеми якості даних: відсутні відгуки, пропущені категорії товарів і можливе дублювання записів під час об'єднання таблиць. Такі особливості потрібно враховувати перед побудовою аналітики та ML-моделей.

Для customer-level features використано `customer_unique_id`, оскільки він ідентифікує реального покупця, тоді як `customer_id` пов'язаний з окремим замовленням. Це дозволяє правильно розраховувати Frequency, Monetary та інші характеристики покупців.

Перед розрахунком `seller_score` дані потрібно було привести до грануляності `seller_id × order_id`, оскільки одне замовлення може містити декілька товарних позицій одного продавця. Без дедуплікації відгуки могли б враховуватися декілька разів і спотворювати середній рейтинг.

**Головний висновок:** правильна грануляність даних, контроль їхньої якості та повторюваність ETL-процесів є основою надійної аналітики й машинного навчання.